In [ ]:
!pip install -U langchain-openai langchain-community langchain-core requests duckduckgo-search

In [ ]:
!pip install "langchain==0.3.27" "langchain-community==0.3.27" "langchain-classic"

INFO: pip is looking at multiple versions of langchain-classic to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-classic to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pydantic_settings-2.16.3-py3-none-any.whl.metadata (4.0 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
ERROR: Cannot install langchain-classic==1.0.0, langchain-classic==1.0.1, langchain-classic==1.0.2, langchain-classic==1.0.3, langchain-classic==1.0.4, langchain-classic==1.0.5, langchain-classic==1.0.6, langchain-classic==1.0.7, langcha

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [ ]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.6 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_google_genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests

In [ ]:
#Step 1: Pick Pull the ReAct prompt LLM Model, Create own

In [ ]:
from google.colab import userdata
weather_data_key=userdata.get('weather_data')

In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={weather_data_key}&query={city}'

  response = requests.get(url)

  return response.json()

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [ ]:
# Initialize Google's Gemini Chat Model
# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",api_key=GOOGLE_API_KEY)

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [ ]:
# Step 2: Pull the ReAct prompt from LangChain Hub or create own
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
Answer the following questions as best you can.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: think about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Question: {input}
Thought: {agent_scratchpad}
""")


In [ ]:
!pip install -U langchain-community tavily-python

In [ ]:
from google.colab import userdata
from langchain_community.tools.tavily_search import TavilySearchResults

# Retrieve Tavily API key
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

# Initialize Tavily
search_tool = TavilySearchResults(max_results=5,tavily_api_key=TAVILY_API_KEY)

# Test search
results = search_tool.invoke("Top news in Chennai today")
print(results)

[{'title': "Chennai News: Today's Latest & Breaking Chennai News Headlines - News18", 'url': 'https://www.news18.com/cities/chennai-news', 'content': "### Tamil Nadu Jail Break: Nine Jail Guards Suspended By Prisons DGP After 17 Undertrials Escape\n ### ‘Flapping Mouth Like A Bengal Monitor Lizard’: TVK Jabs Stalin After DMK Ex-Minister’s Arrest\n ### 'Happy Diwali': Dhanush Keeps Fans Guessing On Political Debut After Meeting 3,000 Women Wing Members\n ### Nayanthara Reunites With Megastar Rajinikanth, Joins Ashwath Marimuthu’s Dharman\n ### Dhanush To Meet 3,000 Women Fans Today Amid Political Debut Buzz, Congress Alleges 'BJP's Operation D'\n ### Chennai Rains: IMD Issues Yellow, Orange Alerts For Several Districts; Heavy Rain Forecast Till October 26\n ### 'No One Needs To Lecture Me On Patriotism': Annamalai Fires Back At BJP After Backing CJP Protests\n ### Tamil Playback Singer Krishnaraj Passes Away at 75; CM Vijay Mourns His Demise [...] ### Tamil Nadu, Puducherry Rain Alert: 

In [ ]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [ ]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [ ]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Tamil Nadu, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Thought: I need to find the capital of Tamil Nadu first, and then check its current weather.
Action: tavily_search_results_json
Action Input: capital of Tamil Nadu[{'title': 'Chennai - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Chennai', 'content': 'Chennai, also known as Madras, is the capital and largest city of Tamil Nadu, the southernmost state of India. It is located on the Coromandel Coast of the Bay of Bengal. As per the 2011 Indian census, Chennai is the sixth-most-populous city in India and forms the fourth-most-populous urban agglomeration. The city is coterminous with Chennai district, which together with the adjoining suburbs constitutes the Chennai Metropolitan Area, one of the largest metropolitan areas of India. It is administered by the Greater Chennai Corporation, which was incorporated in 1688, and is the second oldest municipal corporation in the world after the City of London Corporation. Being the capital of the state,

In [ ]:
response['output']

'The capital of Tamil Nadu is Chennai. The current weather condition in Chennai is a temperature of around 26.7°C (with a feels-like temperature of about 28.3°C), featuring smoky haze, high humidity around 70%, and winds blowing from the north at approximately 16.9 km/h.'